In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# # Install dependencies and ensure the latest version is used
# !pip install --upgrade kagglehub

# import kagglehub
# import os
# import importlib

# # Force reload the module to apply the upgrade without a manual restart
# importlib.reload(kagglehub)

# # Define the dataset identifier
# dataset_id = "nhanayai/shrimpdiseaseimagebd"
# # Define the destination folder in Google Drive
# download_path = "/content/drive/MyDrive/shrimp_disease_images"

# # Create the directory if it doesn't exist
# os.makedirs(download_path, exist_ok=True)

# # Download the latest version to the specified path
# # In the latest kagglehub, dataset_download is the standard method
# path = kagglehub.dataset_download(dataset_id)

# # Move or copy files to the desired Drive location if they aren't already there
# print(f"Dataset downloaded to temporary location: {path}")

# # To specifically ensure it is in your Drive folder:
# import shutil
# if os.path.exists(path):
#     for item in os.listdir(path):
#         s = os.path.join(path, item)
#         d = os.path.join(download_path, item)
#         if os.path.isdir(s):
#             shutil.copytree(s, d, dirs_exist_ok=True)
#         else:
#             shutil.copy2(s, d)

# print(f"Successfully synced dataset to Drive: {download_path}")

In [4]:
cd /content/drive/MyDrive/shrimp_disease_images/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Raw Images/Raw Images

/content/drive/MyDrive/shrimp_disease_images/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Raw Images/Raw Images


In [5]:
ls

'1. Healthy'/  '2. BG'/  '3. WSSV'/  '4. WSSV_BG'/


In [6]:
# !pip install rembg[gpu]

In [7]:
# import os
# import cv2
# import numpy as np
# from rembg import remove
# from PIL import Image
# from tqdm import tqdm

# # Original dataset path
# SOURCE_DIR = r'/content/drive/MyDrive/shrimp_disease_images/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Raw Images/Raw Images'

# # New directory to save the cleaned images
# DEST_DIR = r'/content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images'

# def preprocess_dataset():
#     if not os.path.exists(DEST_DIR):
#         os.makedirs(DEST_DIR)

#     classes = ['1. Healthy', '2. BG', '3. WSSV', '4. WSSV_BG']

#     for cls in classes:
#         src_cls_dir = os.path.join(SOURCE_DIR, cls)
#         dest_cls_dir = os.path.join(DEST_DIR, cls)

#         if not os.path.exists(dest_cls_dir):
#             os.makedirs(dest_cls_dir)

#         if not os.path.isdir(src_cls_dir):
#             continue

#         images = [f for f in os.listdir(src_cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#         print(f"Processing {len(images)} images for class: {cls}")

#         for img_name in tqdm(images):
#             src_path = os.path.join(src_cls_dir, img_name)
#             dest_path = os.path.join(dest_cls_dir, img_name)

#             # Skip if already processed
#             if os.path.exists(dest_path):
#                 continue

#             try:
#                 # Read, remove background, and save
#                 image = cv2.imread(src_path)
#                 image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#                 pil_img = Image.fromarray(image)
#                 clean_img = remove(pil_img).convert('RGB')

#                 # Save back as BGR for CV2 consistency if needed, or save directly via PIL
#                 clean_img.save(dest_path)
#             except Exception as e:
#                 print(f"Error processing {img_name}: {e}")

# # Execute the preprocessing
# preprocess_dataset()
# print(f"Preprocessing complete. Clean images saved to {DEST_DIR}")

## Baseline model comparison with LiteRT export

This section replaces the previous single-model training run with a controlled baseline comparison. The earlier cells still handle Drive mounting, dataset download, and the notebook's existing `rembg` preprocessing into `Processed_Rembg_Images`.

Models are trained one at a time:

- `mobilenet_v3_large`
- `efficientnet_b0`
- `yolo11n-cls`
- `yolo11m-cls`
- `yolo26n-cls`
- `yolo26m-cls`

The comparison follows the benchmark notebook's core training parameters (`224x224`, batch size `32`, AdamW/learning-rate `1e-4` where using PyTorch, `15` epochs, patience `3`) while using this notebook's processed-image input. Validation and test F1 are reported as **macro F1** so each shrimp disease class contributes equally.

A fixed stratified image-level `70/15/15` split is used for every model. This prevents class-ratio drift between models, but it does not guarantee specimen-level leakage protection if several photos of the same shrimp exist under different filenames. Treat results as an image-level baseline unless a specimen/group split is added later.

After each model is trained, the notebook attempts LiteRT/TFLite export. If TFLite conversion fails, it falls back to ONNX and records the export status in the final table.


In [8]:
# Install comparison/export dependencies used by the baseline section.
# Ultralytics is used for YOLO classification training and YOLO TFLite export.
# litert-torch is used for best-effort PyTorch -> LiteRT/TFLite conversion; ONNX is the fallback.
import sys, subprocess, importlib.util

packages = [
    "torchmetrics",
    "ultralytics>=8.3.0",
    "onnx",
    "onnxscript",
    "litert-torch",
]

def import_name_for(package_spec: str) -> str:
    package = package_spec.split(">=")[0].split("==")[0]
    mapping = {
        "litert-torch": "litert_torch",
        "onnxscript": "onnxscript",
    }
    return mapping.get(package, package.replace("-", "_"))

missing = []
for pkg in packages:
    if importlib.util.find_spec(import_name_for(pkg)) is None:
        missing.append(pkg)

if missing:
    print("Installing missing comparison/export dependencies:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("All comparison/export dependencies are already available.")


Installing missing comparison/export dependencies: ['torchmetrics', 'ultralytics>=8.3.0', 'onnx', 'onnxscript', 'litert-torch']


In [9]:
import os
import gc
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import torchvision.models as tv_models
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from ultralytics import YOLO

# ==============================================================================
# 1. Shared configuration
# ==============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-4
NUM_WORKERS = 2

DATA_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images')
OUTPUT_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
EXPORT_DIR = OUTPUT_DIR / 'exports'
YOLO_DATA_DIR = OUTPUT_DIR / 'yolo_dataset'
YOLO_RUNS_DIR = OUTPUT_DIR / 'yolo_runs'
REPORT_DIR = OUTPUT_DIR / 'reports'
for d in [CHECKPOINT_DIR, EXPORT_DIR, YOLO_DATA_DIR, YOLO_RUNS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ['1. Healthy', '2. BG', '3. WSSV', '4. WSSV_BG']
CLASS_NAMES = ['Healthy', 'BG', 'WSSV', 'WSSV_BG']
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = len(CLASS_DIRS)

PYTORCH_MODELS = ['mobilenet_v3_large', 'efficientnet_b0']
YOLO_MODELS = ['yolo11n-cls', 'yolo11m-cls', 'yolo26n-cls', 'yolo26m-cls']
ALL_MODELS = PYTORCH_MODELS + YOLO_MODELS
EXPORT_SMOKE_TEST_ONLY = False  # Set True to test exports without training any baseline models.

# ==============================================================================
# 2. Dataset discovery and fixed stratified split
# ==============================================================================
def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f"Warning: missing class folder: {folder}")
            continue
        for p in sorted(folder.iterdir()):
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                rows.append({
                    'path': str(p),
                    'class_dir': class_dir,
                    'label': CLASS_TO_IDX[class_dir],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No images found under {data_dir}. Run the preprocessing cell first.")
    return df

df = discover_processed_images(DATA_DIR)
print(f"Loaded {len(df)} processed images from {DATA_DIR}")
print(df['class_dir'].value_counts().reindex(CLASS_DIRS))

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df['label'],
    random_state=SEED,
    shuffle=True,
)

for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name}: {len(split_df)} images")
    print(split_df['class_dir'].value_counts().reindex(CLASS_DIRS).to_dict())

# Image-level leakage sanity check. This does not catch same-specimen near duplicates.
train_paths = set(train_df['path'])
val_paths = set(val_df['path'])
test_paths = set(test_df['path'])
assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)
print("Image-level split overlap check passed.")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ShrimpDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    ShrimpDataset(train_df, transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
val_loader = DataLoader(
    ShrimpDataset(val_df, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    ShrimpDataset(test_df, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ==============================================================================
# 3. Shared metric, timing, and export helpers
# ==============================================================================
def macro_f1_metric():
    return torchmetrics.F1Score(task='multiclass', num_classes=NUM_CLASSES, average='macro').to(device)

def count_params(model) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6

def sanitize_name(name: str) -> str:
    return name.replace('/', '_').replace(' ', '_').replace('.', '_')

def build_torchvision_model(model_name: str):
    if model_name == 'mobilenet_v3_large':
        weights = tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        model = tv_models.mobilenet_v3_large(weights=weights)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
    elif model_name == 'efficientnet_b0':
        weights = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = tv_models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    else:
        raise ValueError(f"Unsupported PyTorch model: {model_name}")
    return model.to(device)

def evaluate_pytorch_model(model, loader, criterion=None, timed=False):
    model.eval()
    f1 = macro_f1_metric()
    correct = 0
    total = 0
    total_loss = 0.0

    if timed:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        with torch.no_grad():
            for _ in range(3):
                model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        start = time.time()
    else:
        start = None

    with torch.no_grad():
        for ims, gts in loader:
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = model(ims)
            if criterion is not None:
                total_loss += criterion(logits, gts).item() * ims.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == gts).sum().item()
            total += gts.size(0)
            f1.update(logits, gts)

    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None

    return {
        'loss': total_loss / max(1, total) if criterion is not None else None,
        'accuracy': correct / max(1, total),
        'macro_f1': f1.compute().item(),
        'elapsed': elapsed,
    }

def export_pytorch_model(model, model_name: str):
    model_cpu = model.to('cpu').eval()
    sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    safe_name = sanitize_name(model_name)
    tflite_path = EXPORT_DIR / f'{safe_name}.tflite'
    onnx_path = EXPORT_DIR / f'{safe_name}.onnx'

    tflite_error_text = ''
    try:
        import litert_torch
        if hasattr(litert_torch, 'convert'):
            edge_model = litert_torch.convert(model_cpu, (sample,))
        elif hasattr(litert_torch, 'ai_edge_torch') and hasattr(litert_torch.ai_edge_torch, 'convert'):
            edge_model = litert_torch.ai_edge_torch.convert(model_cpu, (sample,))
        else:
            raise AttributeError('litert_torch does not expose a supported convert() API')

        if hasattr(edge_model, 'export'):
            edge_model.export(str(tflite_path))
        elif hasattr(edge_model, 'save'):
            edge_model.save(str(tflite_path))
        else:
            raise AttributeError('converted LiteRT model has neither export() nor save()')
        return 'tflite', str(tflite_path), ''
    except Exception as tflite_error:
        tflite_error_text = f'{type(tflite_error).__name__}: {tflite_error}'

    try:
        torch.onnx.export(
            model_cpu,
            sample,
            str(onnx_path),
            input_names=['input'],
            output_names=['logits'],
            dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=17,
            dynamo=False,
        )
        return 'onnx', str(onnx_path), f'TFLite failed: {tflite_error_text}'
    except TypeError:
        # Older PyTorch versions do not accept dynamo=False.
        try:
            torch.onnx.export(
                model_cpu,
                sample,
                str(onnx_path),
                input_names=['input'],
                output_names=['logits'],
                dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
                opset_version=17,
            )
            return 'onnx', str(onnx_path), f'TFLite failed: {tflite_error_text}'
        except Exception as onnx_error:
            return 'failed', '', f'TFLite failed: {tflite_error_text}; ONNX failed: {type(onnx_error).__name__}: {onnx_error}'
    except Exception as onnx_error:
        return 'failed', '', f'TFLite failed: {tflite_error_text}; ONNX failed: {type(onnx_error).__name__}: {onnx_error}'

def train_pytorch_baseline(model_name: str) -> dict:
    print('\n' + '=' * 80)
    print(f'Training PyTorch baseline: {model_name}')
    print('=' * 80)

    model = build_torchvision_model(model_name)
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    best_f1 = -1.0
    epochs_no_improve = 0
    safe_name = sanitize_name(model_name)
    best_path = CHECKPOINT_DIR / f'best_{safe_name}.pth'
    train_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for ims, gts in tqdm(train_loader, desc=f'{model_name} epoch {epoch + 1}/{EPOCHS}', leave=False):
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = model(ims)
            loss = criterion(logits, gts)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * ims.size(0)
            train_correct += (torch.argmax(logits, dim=1) == gts).sum().item()
            train_total += gts.size(0)

        val_metrics = evaluate_pytorch_model(model, val_loader, criterion=criterion, timed=False)
        train_acc = train_correct / max(1, train_total)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss / max(1, train_total):.4f} - Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} - Macro F1: {val_metrics['macro_f1']:.4f}"
        )

        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  --> Saved best checkpoint: macro F1 {best_f1:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  --> No improvement ({epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print('  --> Early stopping triggered.')
            break

    train_time = time.time() - train_start
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    params_m = count_params(model)
    test_metrics = evaluate_pytorch_model(model, test_loader, criterion=None, timed=True)
    inf_time = test_metrics['elapsed']
    export_format, export_path, export_note = export_pytorch_model(model, model_name)
    model.to(device)

    result = {
        'Model': model_name,
        'Parameters (M)': round(params_m, 2),
        'Training Time (s)': round(train_time, 1),
        'Val F1-Score': round(best_f1, 4),
        'Test Accuracy': round(test_metrics['accuracy'], 4),
        'Test F1-Score': round(test_metrics['macro_f1'], 4),
        'Inference Time (s)': round(inf_time, 2),
        'FPS': round(len(test_df) / inf_time, 1),
        'Latency (ms)': round((inf_time / len(test_df)) * 1000, 2),
        'Export Format': export_format,
        'Export Path': export_path,
        'Export Note': export_note,
    }

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result

# ==============================================================================
# 4. YOLO split preparation, training, evaluation, and export
# ==============================================================================
def prepare_yolo_dataset():
    if YOLO_DATA_DIR.exists():
        shutil.rmtree(YOLO_DATA_DIR)
    for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        for class_dir in CLASS_DIRS:
            (YOLO_DATA_DIR / split_name / class_dir).mkdir(parents=True, exist_ok=True)
        for _, row in split_df.iterrows():
            src = Path(row['path'])
            dst = YOLO_DATA_DIR / split_name / row['class_dir'] / src.name
            if dst.exists():
                stem = dst.stem
                suffix = dst.suffix
                dst = dst.with_name(f'{stem}_{abs(hash(str(src))) % 10_000_000}{suffix}')
            shutil.copy2(src, dst)
    print(f'Prepared YOLO classification dataset at {YOLO_DATA_DIR}')

prepare_yolo_dataset()

def yolo_device_arg():
    return 0 if torch.cuda.is_available() else 'cpu'

def evaluate_yolo_model(yolo_model, eval_df: pd.DataFrame, timed=False):
    names = yolo_model.names
    name_to_idx = {v: int(k) for k, v in names.items()}
    source_paths = eval_df['path'].tolist()

    if timed:
        # Warm up with one image. Keep this out of the measured interval.
        _ = yolo_model.predict(source=source_paths[:1], imgsz=IMG_SIZE, device=yolo_device_arg(), verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
    else:
        start = None

    preds = yolo_model.predict(
        source=source_paths,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=yolo_device_arg(),
        verbose=False,
    )

    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None

    y_true = torch.tensor([name_to_idx[c] for c in eval_df['class_dir'].tolist()], device=device)
    y_pred = torch.tensor([int(r.probs.top1) for r in preds], device=device)
    logits_for_metric = torch.nn.functional.one_hot(y_pred, num_classes=NUM_CLASSES).float()
    f1 = macro_f1_metric()
    f1.update(logits_for_metric, y_true)
    accuracy = (y_pred == y_true).float().mean().item()

    return {
        'accuracy': accuracy,
        'macro_f1': f1.compute().item(),
        'elapsed': elapsed,
    }

def export_yolo_model(yolo_model, model_name: str):
    try:
        exported = yolo_model.export(format='tflite', imgsz=IMG_SIZE)
        return 'tflite', str(exported), ''
    except Exception as tflite_error:
        try:
            exported = yolo_model.export(format='onnx', imgsz=IMG_SIZE)
            return 'onnx', str(exported), f'TFLite failed: {type(tflite_error).__name__}: {tflite_error}'
        except Exception as onnx_error:
            return 'failed', '', f'TFLite failed: {tflite_error}; ONNX failed: {onnx_error}'

def train_yolo_baseline(model_name: str) -> dict:
    print('\n' + '=' * 80)
    print(f'Training YOLO baseline: {model_name}')
    print('=' * 80)

    weights_name = f'{model_name}.pt'
    train_start = time.time()
    yolo = YOLO(weights_name)
    yolo.train(
        data=str(YOLO_DATA_DIR),
        task='classify',
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        seed=SEED,
        project=str(YOLO_RUNS_DIR),
        name=sanitize_name(model_name),
        exist_ok=True,
        device=yolo_device_arg(),
        verbose=True,
    )
    train_time = time.time() - train_start

    best_path = YOLO_RUNS_DIR / sanitize_name(model_name) / 'weights' / 'best.pt'
    if not best_path.exists():
        # Fallback for Ultralytics version-specific save layouts.
        candidates = sorted((YOLO_RUNS_DIR / sanitize_name(model_name)).glob('**/best.pt'))
        if not candidates:
            raise FileNotFoundError(f'Could not locate YOLO best checkpoint for {model_name}')
        best_path = candidates[-1]

    best_yolo = YOLO(str(best_path))
    val_metrics = evaluate_yolo_model(best_yolo, val_df, timed=False)
    test_metrics = evaluate_yolo_model(best_yolo, test_df, timed=True)
    inf_time = test_metrics['elapsed']
    params_m = count_params(best_yolo.model)
    export_format, export_path, export_note = export_yolo_model(best_yolo, model_name)

    result = {
        'Model': model_name,
        'Parameters (M)': round(params_m, 2),
        'Training Time (s)': round(train_time, 1),
        'Val F1-Score': round(val_metrics['macro_f1'], 4),
        'Test Accuracy': round(test_metrics['accuracy'], 4),
        'Test F1-Score': round(test_metrics['macro_f1'], 4),
        'Inference Time (s)': round(inf_time, 2),
        'FPS': round(len(test_df) / inf_time, 1),
        'Latency (ms)': round((inf_time / len(test_df)) * 1000, 2),
        'Export Format': export_format,
        'Export Path': export_path,
        'Export Note': export_note,
    }

    del yolo, best_yolo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result

# ==============================================================================
# 5. Export smoke tests before long training
# ==============================================================================
def run_export_smoke_tests():
    print('\n' + '=' * 80)
    print('EXPORT SMOKE TESTS')
    print('=' * 80)

    smoke_results = []

    # PyTorch smoke test: verifies LiteRT/TFLite attempt and ONNX fallback before training.
    try:
        smoke_model = build_torchvision_model('mobilenet_v3_large')
        with torch.no_grad():
            _ = smoke_model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device))
        export_format, export_path, export_note = export_pytorch_model(smoke_model, 'smoke_mobilenet_v3_large')
        smoke_model.to(device)
        smoke_results.append({
            'Target': 'PyTorch mobilenet_v3_large',
            'Export Format': export_format,
            'Export Path': export_path,
            'Export Note': export_note,
        })
        del smoke_model
        if export_format == 'failed':
            raise RuntimeError(f'PyTorch export smoke test failed. {export_note}')
    except Exception as exc:
        smoke_results.append({
            'Target': 'PyTorch mobilenet_v3_large',
            'Export Format': 'failed',
            'Export Path': '',
            'Export Note': f'{type(exc).__name__}: {exc}',
        })

    # YOLO smoke test: pretrained tiny classifier export is the fastest way to catch exporter/package issues.
    try:
        smoke_yolo = YOLO('yolo11n-cls.pt')
        export_format, export_path, export_note = export_yolo_model(smoke_yolo, 'smoke_yolo11n-cls')
        smoke_results.append({
            'Target': 'YOLO yolo11n-cls',
            'Export Format': export_format,
            'Export Path': export_path,
            'Export Note': export_note,
        })
        del smoke_yolo
        if export_format == 'failed':
            raise RuntimeError(f'YOLO export smoke test failed. {export_note}')
    except Exception as exc:
        smoke_results.append({
            'Target': 'YOLO yolo11n-cls',
            'Export Format': 'failed',
            'Export Path': '',
            'Export Note': f'{type(exc).__name__}: {exc}',
        })

    smoke_df = pd.DataFrame(smoke_results)
    display(smoke_df)
    smoke_df.to_csv(REPORT_DIR / 'export_smoke_test_results.csv', index=False)

    hard_failures = smoke_df[smoke_df['Export Format'] == 'failed']
    if not hard_failures.empty:
        raise RuntimeError('Export smoke test failed. Fix exporter dependencies before training all models.')

    print('Export smoke tests passed. Continuing to baseline training.')

run_export_smoke_tests()

if EXPORT_SMOKE_TEST_ONLY:
    raise SystemExit('EXPORT_SMOKE_TEST_ONLY=True; stopping after export smoke tests by request.')

# ==============================================================================
# 5. Sequential comparison loop
# ==============================================================================
comparison_results = []
for model_name in ALL_MODELS:
    try:
        if model_name in PYTORCH_MODELS:
            result = train_pytorch_baseline(model_name)
        else:
            result = train_yolo_baseline(model_name)
        comparison_results.append(result)

        partial_df = pd.DataFrame(comparison_results)
        partial_df.to_csv(REPORT_DIR / 'baseline_model_comparison_partial.csv', index=False)
        print('Recorded result:')
        display(pd.DataFrame([result]))
    except Exception as exc:
        print(f'ERROR while running {model_name}: {type(exc).__name__}: {exc}')
        comparison_results.append({
            'Model': model_name,
            'Parameters (M)': np.nan,
            'Training Time (s)': np.nan,
            'Val F1-Score': np.nan,
            'Test Accuracy': np.nan,
            'Test F1-Score': np.nan,
            'Inference Time (s)': np.nan,
            'FPS': np.nan,
            'Latency (ms)': np.nan,
            'Export Format': 'failed',
            'Export Path': '',
            'Export Note': f'Run failed: {type(exc).__name__}: {exc}',
        })
        pd.DataFrame(comparison_results).to_csv(REPORT_DIR / 'baseline_model_comparison_partial.csv', index=False)

# ==============================================================================
# 6. Final table
# ==============================================================================
df_summary = pd.DataFrame(comparison_results)
if not df_summary.empty:
    df_summary = df_summary.sort_values(by='Test F1-Score', ascending=False, na_position='last').reset_index(drop=True)
    print('\n' + '=' * 80)
    print('BASELINE PERFORMANCE SUMMARY: SHRIMP DISEASE CLASSIFICATION')
    print('=' * 80)
    display(df_summary)
    csv_path = REPORT_DIR / 'baseline_model_comparison_summary.csv'
    json_path = REPORT_DIR / 'baseline_model_comparison_summary.json'
    df_summary.to_csv(csv_path, index=False)
    df_summary.to_json(json_path, orient='records', indent=2)
    print(f'Saved summary CSV: {csv_path}')
    print(f'Saved summary JSON: {json_path}')
else:
    print('No model results were produced.')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Loaded 1149 processed images from /content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images
class_dir
1. Healthy    403
2. BG         198
3. WSSV       328
4. WSSV_BG    220
Name: count, dtype: int64
train: 804 images
{'1. Healthy': 282, '2. BG': 139, '3. WSSV': 229, '4. WSSV_BG': 154}
val: 172 images
{'1. Healthy': 60, '2. BG': 30, '3. WSSV': 49, '4. WSSV_BG': 33}
test: 173 images
{'1. Healthy': 61, '2. BG': 29, '3. WSSV': 50, '4. WSSV_BG': 33}
Image-level split overlap check passed.
Prepared YOLO classification dataset at /content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/yolo_dataset

EXPORT SMOKE TESTS
Downloading: "https://down

100%|██████████| 21.1M/21.1M [00:00<00:00, 80.2MB/s]
/tmp/ipykernel_2328/1387526105.py:264: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n-cls summary (fused): 47 layers, 2,807,024 parameters, 0 gradients, 4.2 GFLOPs

PyTorch: starting from 'yolo11n-cls.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 1000) (5.5 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 277ms
Prepared 3 packages in 2.77s
Installed 3 packages in 11ms
 + colorama==0.4.6
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 3.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 5.5s, saved as 

,Target,Export Format,Export Path,Export Note
0,PyTorch mobilenet_v3_large,onnx,/content/drive/MyDrive/shrimp_disease_images/b...,TFLite failed: ModuleNotFoundError: No module ...
1,YOLO yolo11n-cls,tflite,yolo11n-cls_saved_model/yolo11n-cls_float32.tf...,


Export smoke tests passed. Continuing to baseline training.

Training PyTorch baseline: mobilenet_v3_large


...


Training YOLO baseline: yolo26m-cls
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m-cls

,Model,Parameters (M),Training Time (s),Val F1-Score,Test Accuracy,Test F1-Score,Inference Time (s),FPS,Latency (ms),Export Format,Export Path,Export Note
0,yolo26m-cls,10.35,218.8,0.6074,0.6185,0.5913,16.87,10.3,97.52,tflite,/content/drive/MyDrive/shrimp_disease_images/b...,



BASELINE PERFORMANCE SUMMARY: SHRIMP DISEASE CLASSIFICATION


,Model,Parameters (M),Training Time (s),Val F1-Score,Test Accuracy,Test F1-Score,Inference Time (s),FPS,Latency (ms),Export Format,Export Path,Export Note
0,yolo26n-cls,1.53,680.4,0.7742,0.8150,0.8073,17.93,9.7,103.62,tflite,/content/drive/MyDrive/shrimp_disease_images/b...,
1,efficientnet_b0,4.01,512.4,0.7782,0.7399,0.7232,8.05,21.5,46.51,onnx,/content/drive/MyDrive/shrimp_disease_images/b...,TFLite failed: ModuleNotFoundError: No module ...
2,yolo11n-cls,1.53,253.0,0.6360,0.6416,0.6226,17.79,9.7,102.84,tflite,/content/drive/MyDrive/shrimp_disease_images/b...,
3,yolo26m-cls,10.35,218.8,0.6074,0.6185,0.5913,16.87,10.3,97.52,tflite,/content/drive/MyDrive/shrimp_disease_images/b...,
4,yolo11m-cls,10.35,216.5,0.5309,0.6590,0.5600,16.08,10.8,92.94,tflite,/content/drive/MyDrive/shrimp_disease_images/b...,
5,mobilenet_v3_large,4.21,159.4,0.4293,0.4855,0.3906,6.14,28.2,35.51,onnx,/content/drive/MyDrive/shrimp_disease_images/b...,TFLite failed: ModuleNotFoundError: No module ...


Saved summary CSV: /content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/reports/baseline_model_comparison_summary.csv
Saved summary JSON: /content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/reports/baseline_model_comparison_summary.json


## Notes on interpreting this table

- `Val F1-Score` and `Test F1-Score` are **macro F1**, not micro F1. Macro F1 is more appropriate here because the shrimp disease classes are imbalanced and each class should contribute equally.
- `Training Time (s)` includes only the model's training phase. Export time is not included.
- `Inference Time (s)`, `FPS`, and `Latency (ms)` are measured over the shared test split after a short warm-up.
- `Export Format` reports `tflite` when LiteRT/TFLite export succeeds and `onnx` when the notebook falls back to ONNX. Check `Export Note` for conversion failures.
- The split is stratified and image-level. If specimen identifiers become available, rerun with a grouped split to better control leakage between photos of the same shrimp.
